# Agentic RAG with LangGraph and MCP

This notebook runs the library blocks of the lessons *Build the graph with LangGraph* and *Knowledge tools and MCP*. No GPU, no API key and no language model are needed: routing, grading, rewriting and answering are rule-based stand-ins. The outputs saved in the cells come from a real run with the pinned versions.

In [ ]:
!pip install -q langgraph==1.2.24 mcp==2.3.0
print('installed')

## 1. The RAG graph with LangGraph 1.2.24

In [ ]:
PASSAGES = [
    {"id": "hr-leave",    "source": "hr",       "text": "Full-time employees receive 24 days of paid annual leave per calendar year."},
    {"id": "hr-sick",     "source": "hr",       "text": "Employees may take up to 10 paid sick days per year without a medical certificate."},
    {"id": "hr-carry",    "source": "hr",       "text": "Up to 5 unused annual leave days may be carried over and must be used by 31 March."},
    {"id": "hr-parental", "source": "hr",       "text": "The primary caregiver receives 26 weeks of paid parental leave."},
    {"id": "faq-seats",   "source": "product",  "text": "The Starter plan includes up to 5 seats."},
    {"id": "faq-export",  "source": "product",  "text": "Admins can export all tickets as CSV from Settings, then Data."},
    {"id": "faq-trash",   "source": "product",  "text": "Deleted tickets stay in the trash for 30 days and are then removed for good."},
    {"id": "faq-region",  "source": "product",  "text": "Customer data is stored in the Mumbai region by default."},
    {"id": "faq-sso",     "source": "security", "text": "Single sign-on with SAML is available on the Business plan and above."},
    {"id": "sec-password","source": "security", "text": "Passwords must have at least 14 characters."},
    {"id": "run-sev1",    "source": "support",  "text": "A Sev 1 incident needs a first response within 5 minutes."},
]

import math, re
STOPWORDS = set("a an and are at be by can do does for from get have how i in is it many may me must my of on or our "
                "the their then to up we what when where which who with you".split())
TEXT = {p["id"]: p["text"] for p in PASSAGES}

def terms(text):
    """lower-case words without filler words; a final s is cut so 'tickets' matches 'ticket'"""
    words = re.findall(r"[a-z0-9]+", text.lower())
    return {w[:-1] if len(w) > 3 and w.endswith("s") else w for w in words if w not in STOPWORDS}

def search(query, k=3, source=None):
    """stand-in for embedding search: shared words, rare words weigh more"""
    wanted, scored = terms(query), []
    for p in PASSAGES:
        shared = wanted & terms(p["text"])
        if shared and source in (None, p["source"]):
            weight = sum(math.log(1 + len(PASSAGES) / sum(w in terms(q["text"]) for q in PASSAGES)) for w in shared)
            scored.append((p["id"], round(weight, 2)))
    return sorted(scored, key=lambda pair: (-pair[1], pair[0]))[:k]

GRADE_MIN = 0.5                        # ASSUMED: share of the question's words a passage must contain
SYNONYMS = {"holiday": "annual leave", "vacation": "annual leave", "ill": "sick", "download": "export",
            "bin": "trash", "erased": "deleted", "users": "seats", "login": "sign"}

def route(question):
    """stand-in for a model that picks what to do with a question"""
    text = question.lower()
    if re.match(r"(hi|hello|thanks|thank you)\b", text):
        return "direct"
    if re.search(r"\d\s*[+*/-]\s*\d", text):
        return "calculate"
    if len(terms(question)) < 2:
        return "clarify"
    return "search"

def grade(question, passage_text):
    wanted = terms(question)
    return bool(wanted) and len(wanted & terms(passage_text)) / len(wanted) >= GRADE_MIN

def rewrite(query):
    return " ".join(SYNONYMS.get(w.strip("?.,").lower(), w) for w in query.split())

def generate(question, ids):
    return f"{TEXT[ids[0]]} [{ids[0]}]" if ids else "I do not know."

import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    question: str                              # what the user asked
    query: str                                 # what is searched
    found: list
    docs: list
    answer: str
    rewrites: int
    path: Annotated[list, operator.add]        # reducer: new items are appended

# a node reads the state and returns only the keys it changes
def retrieve(state):
    return {"found": [i for i, _ in search(state["query"])], "path": ["retrieve"]}

def grade_node(state):
    return {"docs": [i for i in state["found"] if grade(state["query"], TEXT[i])], "path": ["grade"]}

def rewrite_node(state):
    return {"query": rewrite(state["query"]), "rewrites": state["rewrites"] + 1, "path": ["rewrite"]}

def answer_node(state):
    return {"answer": generate(state["question"], state["docs"]), "path": ["answer"]}

def route_node(state):
    reply = {"direct": "Hello. Ask me about Orbit Labs.", "calculate": "Use the calculator.",
             "clarify": "What exactly do you mean?"}[route(state["question"])]
    return {"answer": reply, "path": ["route"]}

# edge functions return the name of the next node
def pick_start(state):
    return "retrieve" if route(state["question"]) == "search" else "route"

def after_grade(state):
    can_try = state["rewrites"] < 1 and rewrite(state["query"]) != state["query"]
    return "rewrite" if not state["docs"] and can_try else "answer"

builder = StateGraph(State)
for name, node in (("retrieve", retrieve), ("grade", grade_node), ("rewrite", rewrite_node),
                   ("answer", answer_node), ("route", route_node)):
    builder.add_node(name, node)
builder.add_conditional_edges(START, pick_start, ["retrieve", "route"])
builder.add_edge("retrieve", "grade")
builder.add_conditional_edges("grade", after_grade, ["rewrite", "answer"])
builder.add_edge("rewrite", "retrieve")
builder.add_edge("answer", END)
builder.add_edge("route", END)
graph = builder.compile()

QUESTIONS = ['How many days of annual leave do employees get?', 'Does vacation carry over?', 'How long do deleted tickets stay?', 'How many users does the Starter plan include?', 'What is the dress code for a Sev 1 call?', 'Hello there']
for question in QUESTIONS:
    start = {"question": question, "query": question, "rewrites": 0, "path": []}
    final = graph.invoke(start, config={"recursion_limit": 10})
    print(question)
    print("  ", " > ".join(final["path"]), "|", final["answer"][-14:])

How many days of annual leave do employees get?
   retrieve > grade > answer | ar. [hr-leave]
Does vacation carry over?
   retrieve > grade > rewrite > retrieve > grade > answer | ch. [hr-carry]
How long do deleted tickets stay?
   retrieve > grade > answer | d. [faq-trash]
How many users does the Starter plan include?
   retrieve > grade > answer | s. [faq-seats]
What is the dress code for a Sev 1 call?
   retrieve > grade > answer | I do not know.
Hello there
   route | ut Orbit Labs.


## 2. Streaming a run

In [ ]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    tries: int
    log: Annotated[list, operator.add]     # appended
    last: str                              # no reducer: replaced

def search(state):
    return {"tries": state["tries"] + 1, "log": ["search"], "last": "search"}

def check(state):
    return {"log": ["check"], "last": "check", "typo_key": 1}

builder = StateGraph(State)
builder.add_node("search", search)
builder.add_node("check", check)
builder.add_edge(START, "search")
builder.add_edge("search", "check")
builder.add_conditional_edges("check", lambda s: "search" if s["tries"] < 2 else END, ["search", END])
graph = builder.compile()
start = {"tries": 0, "log": [], "last": ""}

print("updates:")
for event in graph.stream(start, stream_mode="updates"):
    print("  ", event)
print("values:")
for snapshot in graph.stream(start, stream_mode="values"):
    print("  ", snapshot)

updates:
   {'search': {'tries': 1, 'log': ['search'], 'last': 'search'}}
   {'check': {'log': ['check'], 'last': 'check'}}
   {'search': {'tries': 2, 'log': ['search'], 'last': 'search'}}
   {'check': {'log': ['check'], 'last': 'check'}}
values:
   {'tries': 0, 'log': [], 'last': ''}
   {'tries': 1, 'log': ['search'], 'last': 'search'}
   {'tries': 1, 'log': ['search', 'check'], 'last': 'check'}
   {'tries': 2, 'log': ['search', 'check', 'search'], 'last': 'search'}
   {'tries': 2, 'log': ['search', 'check', 'search', 'check'], 'last': 'check'}


## 3. The recursion limit

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START
from langgraph.errors import GraphRecursionError

class State(TypedDict):
    tries: int

builder = StateGraph(State)
builder.add_node("again", lambda s: {"tries": s["tries"] + 1})
builder.add_edge(START, "again")
builder.add_edge("again", "again")          # no way out
graph = builder.compile()

runs = []
for config in ({"recursion_limit": 5}, {}):
    runs.clear()
    try:
        for snapshot in graph.stream({"tries": 0}, config=config, stream_mode="values"):
            runs.append(snapshot["tries"])
    except GraphRecursionError as error:
        print("config", config, "-> stopped after", runs[-1], "node runs")
        print("  ", str(error).splitlines()[0][:60])

builder.add_edge("again", "nowhere")
try:
    builder.compile()
except ValueError as error:
    print("compile:", error)

config {'recursion_limit': 5} -> stopped after 5 node runs
   Recursion limit of 5 reached without hitting a stop conditio
config {} -> stopped after 10007 node runs
   Recursion limit of 10007 reached without hitting a stop cond
compile: Found edge ending at unknown node `nowhere`


## 4. A knowledge tool with mcp 2.3.0, in-process

The last line of this cell uses `await`, because a notebook already runs an event loop.

In [ ]:
PASSAGES = [
    {"id": "hr-leave",    "source": "hr",       "text": "Full-time employees receive 24 days of paid annual leave per calendar year."},
    {"id": "hr-sick",     "source": "hr",       "text": "Employees may take up to 10 paid sick days per year without a medical certificate."},
    {"id": "hr-carry",    "source": "hr",       "text": "Up to 5 unused annual leave days may be carried over and must be used by 31 March."},
    {"id": "hr-parental", "source": "hr",       "text": "The primary caregiver receives 26 weeks of paid parental leave."},
    {"id": "faq-seats",   "source": "product",  "text": "The Starter plan includes up to 5 seats."},
    {"id": "faq-export",  "source": "product",  "text": "Admins can export all tickets as CSV from Settings, then Data."},
    {"id": "faq-trash",   "source": "product",  "text": "Deleted tickets stay in the trash for 30 days and are then removed for good."},
    {"id": "faq-region",  "source": "product",  "text": "Customer data is stored in the Mumbai region by default."},
    {"id": "faq-sso",     "source": "security", "text": "Single sign-on with SAML is available on the Business plan and above."},
    {"id": "sec-password","source": "security", "text": "Passwords must have at least 14 characters."},
    {"id": "run-sev1",    "source": "support",  "text": "A Sev 1 incident needs a first response within 5 minutes."},
]

import math, re
STOPWORDS = set("a an and are at be by can do does for from get have how i in is it many may me must my of on or our "
                "the their then to up we what when where which who with you".split())
TEXT = {p["id"]: p["text"] for p in PASSAGES}

def terms(text):
    """lower-case words without filler words; a final s is cut so 'tickets' matches 'ticket'"""
    words = re.findall(r"[a-z0-9]+", text.lower())
    return {w[:-1] if len(w) > 3 and w.endswith("s") else w for w in words if w not in STOPWORDS}

def search(query, k=3, source=None):
    """stand-in for embedding search: shared words, rare words weigh more"""
    wanted, scored = terms(query), []
    for p in PASSAGES:
        shared = wanted & terms(p["text"])
        if shared and source in (None, p["source"]):
            weight = sum(math.log(1 + len(PASSAGES) / sum(w in terms(q["text"]) for q in PASSAGES)) for w in shared)
            scored.append((p["id"], round(weight, 2)))
    return sorted(scored, key=lambda pair: (-pair[1], pair[0]))[:k]

import asyncio
import json
from typing import Annotated
from pydantic import Field
from mcp import Client
from mcp.server.mcpserver import MCPServer
from mcp.server.mcpserver.exceptions import ToolError

def make_server(allowed_sources):
    """one server object for one authenticated caller: access is fixed here, outside the arguments"""
    server = MCPServer("orbit-knowledge", version="1.0.0", log_level="CRITICAL")

    @server.tool(description="Search Orbit Labs documents. Returns passages with their ids.")
    def search_knowledge(query: str, limit: Annotated[int, Field(ge=1, le=3)] = 2) -> list[dict]:
        if not query.strip():
            raise ToolError("query is empty")
        hits = [i for i, _ in search(query, k=len(PASSAGES))]
        source = {p["id"]: p["source"] for p in PASSAGES}
        return [{"id": i, "text": TEXT[i]} for i in hits if source[i] in allowed_sources][:limit]

    return server

async def main():
    async with Client(make_server({"product", "security"})) as client:
        print("protocol", client.protocol_version, "| server", client.server_info.name, client.server_info.version)
        tool = (await client.list_tools()).tools[0]
        print(tool.name, "| required:", tool.input_schema["required"])
        print("arguments:", list(tool.input_schema["properties"]))
        calls = [("good", "search_knowledge", {"query": "export tickets", "limit": 1}),
                 ("hidden source", "search_knowledge", {"query": "annual leave"}),
                 ("extra arg", "search_knowledge", {"query": "annual leave", "role": "employee"}),
                 ("limit 50", "search_knowledge", {"query": "export", "limit": 50}),
                 ("empty query", "search_knowledge", {"query": " "}),
                 ("wrong name", "delete_everything", {})]
        for label, name, arguments in calls:
            result = await client.call_tool(name, arguments)
            if result.is_error:
                lines = result.content[0].text.splitlines()
                shown = lines[0] if len(lines) == 1 else f"{lines[1]}: {lines[2].split('[')[0].strip()}"
            else:
                shown = json.dumps([hit["id"] for hit in result.structured_content["result"]])
            print(f"{label:13} error={result.is_error!s:5} {shown[:62]}")

await main()

protocol 2026-07-28 | server orbit-knowledge 1.0.0
search_knowledge | required: ['query']
arguments: ['query', 'limit']
good          error=False ["faq-export"]
hidden source error=False []
extra arg     error=False []
limit 50      error=True  limit: Input should be less than or equal to 3
empty query   error=True  Error executing tool search_knowledge: query is empty
wrong name    error=True  Unknown tool: delete_everything


## Try next

Change the step budget, the questions and the grading threshold. Never paste a secret into a notebook cell.